In [1]:
# Added for the public reproduction package: load the pseudonymized CSV instead of the
# private spreadsheet and write any output files to ../output/notebooks/. See repro_data.py.
import repro_data


In [2]:
import pandas as pd
from scipy import stats

# ==========================
# Configuración
# ==========================

ARCHIVO = "Wallet-pattern-.xlsx"
HOJA = "Datos cuanti"
COL_VALOR = "SUS UX"

# ==========================
# Cargar datos
# ==========================

df = repro_data.read_datos_cuanti()
df.columns = [str(c).strip() for c in df.columns]

sus = pd.to_numeric(df[COL_VALOR], errors="coerce").dropna()

# ==========================
# Shapiro-Wilk
# ==========================

W, p_shapiro = stats.shapiro(sus)

# ==========================
# Estadísticos descriptivos
# ==========================

n = len(sus)
media = sus.mean()
mediana = sus.median()
desvio = sus.std(ddof=1)

# Asimetría y curtosis
skewness = stats.skew(sus, bias=False)
kurtosis = stats.kurtosis(sus, fisher=True, bias=False)

# Test de curtosis
_, p_kurtosis = stats.kurtosistest(sus)

# ==========================
# Outliers (criterio IQR)
# ==========================

Q1 = sus.quantile(0.25)
Q3 = sus.quantile(0.75)
IQR = Q3 - Q1

lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers = sorted(sus[(sus < lim_inf) | (sus > lim_sup)])

# ==========================
# Imprimir resultados
# ==========================

print("Shapiro-Wilk")
print("-------------------------------")
print(f"P-value: {p_shapiro:.8f}")
print(f"W: {W:.4f}")
print(f"Sample size (n): {n}")
print(f"Average: {media:.3f}")
print(f"Median: {mediana:.1f}")
print(f"Sample standard deviation: {desvio:.4f}")
print(f"Skewness: {skewness:.4f}")
print(f"Excess kurtosis: {kurtosis:.4f}")
print(f"Kurtosis p-value: {p_kurtosis:.4f}")

print("\nOutliers (IQR):")
if len(outliers) == 0:
    print("None")
else:
    print(", ".join(f"{x:.1f}" for x in outliers))

# ==========================
# Generar tabla LaTeX
# ==========================

skew_dir = "left/negative" if skewness < 0 else "right/positive"
kurt_desc = "long heavy tails" if kurtosis > 0 else "light tails"

outliers_tex = (
    ", ".join(f"{x:.1f}" for x in outliers)
    if len(outliers) > 0
    else "None"
)

latex = f"""
\\begin{{table}}[tbp]
\\centering
\\begin{{tabular}}{{@{{}}ll@{{}}}}
\\toprule
Parameter & Value \\\\ \\midrule
P-value & {p_shapiro:.8f} \\\\
W & {W:.4f} \\\\
Sample size (n) & {n} \\\\
Average ($\\bar{{x}}$) & {media:.3f} \\\\
Median & {mediana:.1f} \\\\
Sample standard deviation (S) & {desvio:.4f} \\\\
Skewness & {skewness:.4f} ({skew_dir}) \\\\
Excess kurtosis & {kurtosis:.4f} ({kurt_desc}, $p={p_kurtosis:.3f}$) \\\\
Outliers & {outliers_tex} \\\\
\\bottomrule
\\end{{tabular}}
\\caption{{Shapiro-Wilk test of the SUS score sample.}}
\\label{{tab:sus-shapiro-wilk}}
\\end{{table}}
"""

print("\n")
print("=" * 70)
print("TABLA LATEX")
print("=" * 70)
print(latex)

with open("tabla_shapiro.tex", "w", encoding="utf-8") as f:
    f.write(latex)

print("\nArchivo guardado como: tabla_shapiro.tex")

Shapiro-Wilk
-------------------------------
P-value: 0.00018478
W: 0.9131
Sample size (n): 67
Average: 72.612
Median: 77.5
Sample standard deviation: 17.2297
Skewness: -1.1621
Excess kurtosis: 1.4403
Kurtosis p-value: 0.0484

Outliers (IQR):
17.5, 22.5


TABLA LATEX

\begin{table}[tbp]
\centering
\begin{tabular}{@{}ll@{}}
\toprule
Parameter & Value \\ \midrule
P-value & 0.00018478 \\
W & 0.9131 \\
Sample size (n) & 67 \\
Average ($\bar{x}$) & 72.612 \\
Median & 77.5 \\
Sample standard deviation (S) & 17.2297 \\
Skewness & -1.1621 (left/negative) \\
Excess kurtosis & 1.4403 (long heavy tails, $p=0.048$) \\
Outliers & 17.5, 22.5 \\
\bottomrule
\end{tabular}
\caption{Shapiro-Wilk test of the SUS score sample.}
\label{tab:sus-shapiro-wilk}
\end{table}


Archivo guardado como: tabla_shapiro.tex
